# Klasifikasi Kategori Kedalaman Lubang Jalan: YOLO26 vs RT-DETR (versi Kaggle, v2)

**v2 menambah: uji kondisi buatan (8), training ulang dengan augmentasi kuat (9-10), dan analisis citra berisi banyak lubang (11).** Model baseline tidak dilatih ulang. Model diambil dari output versi notebook sebelumnya.

Sebelum menjalankan, atur di panel kanan:
1. **Input > Add Input**: dataset `PothRGBD` harus tetap terpasang.
2. **Input > Add Input > Your Work (atau tab Notebooks)**: cari notebook ini sendiri, pilih versi yang kemarin sukses, klik **+**.
   Ini supaya file model `best.pt` dari kemarin ikut terbaca. Tanpa ini, ketiga model akan dilatih ulang (berjam-jam).
3. **Settings > Accelerator > GPU T4 x2**, dan **Settings > Internet > On**.

Lalu **Save Version > Save & Run All (Commit)**. Hasil baru: `pothole_robustness.zip` di tab **Output**. Perkiraan total 3-4 jam karena ada training ulang dengan augmentasi kuat (bagian 9). Perkiraan total 3-4 jam (karena ada training ulang dengan augmentasi kuat).


## 1. Pasang library dan cari dataset

In [ ]:
!pip -q install ultralytics
import torch, torch._utils, ultralytics
print("ultralytics", ultralytics.__version__, "| torch", torch.__version__,
      "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "TIDAK ADA")
assert torch.cuda.is_available(), "GPU tidak aktif. Settings > Accelerator > GPU T4 x2, lalu jalankan ulang."

In [ ]:
import os, glob, collections, re, shutil
import numpy as np, cv2, pandas as pd

DATA_ROOT = "/kaggle/input"      # semua dataset yang ditempel lewat Add Input ada di sini
WORK = "/kaggle/working"
OUT = f"{WORK}/out"; os.makedirs(OUT, exist_ok=True)
assert os.listdir(DATA_ROOT), "Belum ada dataset. Panel kanan > Input > Add Input > cari PothRGBD."

# Output versi notebook sebelumnya (berisi runs/<model>/weights/best.pt), kalau sudah ditempel lewat Add Input
_prev = sorted(glob.glob(f"{DATA_ROOT}/**/runs/YOLO26s/weights/best.pt", recursive=True))
PREV_RUNS = os.path.dirname(os.path.dirname(os.path.dirname(_prev[0]))) if _prev else None
PREV_ROOT = os.path.dirname(PREV_RUNS) if PREV_RUNS else None
print("Model dari versi sebelumnya:", PREV_RUNS or "TIDAK KETEMU (model akan dilatih ulang)")


## 2. Periksa struktur dataset

In [ ]:
IMG_EXT = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}
summary = collections.defaultdict(collections.Counter)
for d, _, files in os.walk(DATA_ROOT):
    if PREV_ROOT and d.startswith(PREV_ROOT):
        continue
    for f in files:
        summary[d][os.path.splitext(f)[1].lower()] += 1
for d in sorted(summary):
    print(d, dict(summary[d]))

def guess(keys, exts):
    c = [d for d in summary if any(k in os.path.basename(d).lower() for k in keys) and any(summary[d][e] for e in exts)]
    return sorted(c, key=lambda d: -sum(summary[d].values()))

DEPTH_DIRS = guess(["depth"], IMG_EXT | {".npy"})
LABEL_DIRS = [d for d in summary if summary[d][".txt"] > 5]
RGB_DIRS = [d for d in guess(["rgb", "image", "color"], IMG_EXT) if d not in DEPTH_DIRS]
print("\nRGB  :", RGB_DIRS); print("DEPTH:", DEPTH_DIRS); print("LABEL:", LABEL_DIRS)
assert RGB_DIRS and DEPTH_DIRS and LABEL_DIRS, "Folder RGB/depth/label tidak ketemu. Kirim output sel ini ke Claude."

def load_depth(path):
    return np.load(path) if path.endswith(".npy") else cv2.imread(path, cv2.IMREAD_UNCHANGED)

sample = sorted(glob.glob(os.path.join(DEPTH_DIRS[0], "*")))[0]
d = load_depth(sample); nz = d[d > 0]
print(f"\nContoh depth: {os.path.basename(sample)} | shape={d.shape} dtype={d.dtype} "
      f"| min>0={nz.min() if nz.size else None} median={np.median(nz) if nz.size else None} max={d.max()}")

## 3. Hitung kedalaman dan buat label kategori

Kategori: **shallow < 25 mm, medium 25-50 mm, deep > 50 mm**.
Gambar dibuang kalau pasangan depth-nya ambigu (nama kembar), kalau ada lubang yang depth-nya tidak valid,
atau kalau ada lubang yang terukur di atas 150 mm (dianggap salah baca sensor).

In [ ]:
import numpy as np
import cv2

# Batas kategori kedalaman (mm): < 25 = shallow, 25-50 = medium, > 50 = deep
THRESH_MM = (25.0, 50.0)
CLASS_NAMES = ["shallow", "medium", "deep"]


def depth_to_class(d_mm):
    """Ubah kedalaman (mm) jadi id kelas 0/1/2."""
    if d_mm < THRESH_MM[0]:
        return 0
    if d_mm <= THRESH_MM[1]:
        return 1
    return 2


def read_polygons(label_path):
    """Baca label segmentasi YOLO: 'cls x1 y1 x2 y2 ...' (koordinat ternormalisasi)."""
    polys = []
    with open(label_path) as f:
        for line in f:
            p = line.split()
            if len(p) < 7 or len(p[1:]) % 2:
                continue
            polys.append(np.array(p[1:], dtype=float).reshape(-1, 2))
    return polys


def fit_plane(xs, ys, zs, iters=3, trim=2.5):
    """Fit bidang z = a*x + b*y + c secara robust (buang outlier tiap iterasi)."""
    A = np.c_[xs, ys, np.ones(len(xs))].astype(float)
    zs = zs.astype(float)
    keep = np.ones(len(zs), bool)
    coef = None
    for _ in range(iters):
        coef, *_ = np.linalg.lstsq(A[keep], zs[keep], rcond=None)
        r = zs - A @ coef
        mad = np.median(np.abs(r[keep] - np.median(r[keep]))) + 1e-6
        new_keep = np.abs(r) < trim * 1.4826 * mad
        if new_keep.sum() < 50:
            break
        keep = new_keep
    return coef


def pothole_depth_mm(depth_mm, poly_norm, ring_px=25, gap_px=5, pct=95):
    """Kedalaman satu lubang (mm).

    Permukaan jalan di sekitar lubang (cincin di luar mask) di-fit jadi bidang,
    lalu kedalaman = persentil ke-95 selisih antara depth di dalam mask dan bidang itu.
    Return None kalau piksel depth yang valid terlalu sedikit.
    """
    h, w = depth_mm.shape
    pts = np.round(poly_norm * [w, h]).astype(np.int32)
    mask = np.zeros((h, w), np.uint8)
    cv2.fillPoly(mask, [pts], 1)

    def k(r):
        return cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * r + 1, 2 * r + 1))

    inner = cv2.dilate(mask, k(gap_px))
    outer = cv2.dilate(mask, k(gap_px + ring_px))
    ring = (outer == 1) & (inner == 0)
    valid = np.isfinite(depth_mm) & (depth_mm > 0)
    ry, rx = np.nonzero(ring & valid)
    my, mx = np.nonzero((mask == 1) & valid)
    if len(ry) < 100 or len(my) < 30:
        return None
    coef = fit_plane(rx, ry, depth_mm[ry, rx])
    resid = depth_mm[my, mx] - (coef[0] * mx + coef[1] * my + coef[2])
    return float(max(np.percentile(resid, pct), 0.0))


def poly_to_bbox(poly_norm):
    """Polygon ternormalisasi -> bbox YOLO (cx, cy, w, h) ternormalisasi."""
    p = np.clip(poly_norm, 0, 1)
    x0, y0 = p.min(0)
    x1, y1 = p.max(0)
    return (x0 + x1) / 2, (y0 + y1) / 2, x1 - x0, y1 - y0

In [ ]:
DEPTH_SCALE_MM = 1.0     # depth RealSense dalam mm
MAX_VALID_MM = 150.0     # di atas ini dianggap salah baca sensor

def index_files(dirs, exts):
    out = {}
    for d in dirs:
        for p in glob.glob(os.path.join(d, "*")):
            stem, ext = os.path.splitext(os.path.basename(p))
            if ext.lower() in exts:
                out[stem] = p
    return out

def file_id(stem):
    """Bagian nama sebelum '_color' / '_depth', mis. '20250227_135438'."""
    return re.split(r"_(?:color|depth)", stem, flags=re.I)[0]

rgb = index_files(RGB_DIRS, IMG_EXT)
dep_raw = index_files(DEPTH_DIRS, IMG_EXT | {".npy"})
lab_raw = index_files(LABEL_DIRS, {".txt"})
n_total = len(rgb)

dep_ids = collections.Counter(file_id(s) for s in dep_raw)
rgb_ids = collections.Counter(file_id(s) for s in rgb)
dup = {i for i, n in dep_ids.items() if n > 1} | {i for i, n in rgb_ids.items() if n > 1}
rgb = {s: p for s, p in rgb.items() if file_id(s) not in dup}
dep_raw = {s: p for s, p in dep_raw.items() if file_id(s) not in dup}
dep_by_id = {file_id(s): p for s, p in dep_raw.items()}
dep = {s: dep_by_id[file_id(s)] for s in rgb if file_id(s) in dep_by_id}
lab = {s: lab_raw[s] for s in rgb if s in lab_raw}
stems = sorted(set(dep) & set(lab))
assert stems, "Tidak ada file yang berpasangan. Kirim output sel 2 ke Claude."

rows, bad_invalid, bad_outlier, empty = [], set(), set(), set()
for s in stems:
    d = load_depth(dep[s]).astype(np.float32)
    assert d.ndim == 2, "Depth bukan 1-channel; kirim output sel 2 ke Claude."
    d *= DEPTH_SCALE_MM
    polys = read_polygons(lab[s])
    if not polys:
        empty.add(s)
    for i, poly in enumerate(polys):
        dm = pothole_depth_mm(d, poly)
        if dm is None:
            bad_invalid.add(s)
        elif dm > MAX_VALID_MM:
            bad_outlier.add(s)
        else:
            rows.append(dict(image=s, idx=i, depth_mm=dm, cls=depth_to_class(dm), bbox=poly_to_bbox(poly)))
df = pd.DataFrame(rows)
df = df[~df.image.isin(bad_invalid | bad_outlier)].reset_index(drop=True)

excl = {"images_total": n_total, "excluded_ambiguous_pairing": n_total - len(rgb),
        "excluded_unpaired": len(rgb) - len(stems), "excluded_no_polygon": len(empty),
        "excluded_invalid_depth": len(bad_invalid), "excluded_depth_over_150mm": len(bad_outlier - bad_invalid),
        "images_used": df.image.nunique(), "potholes_used": len(df)}
pd.Series(excl).to_csv(f"{OUT}/dataset_filtering.csv", header=["count"])
print(pd.Series(excl).to_string())

### Sebaran kelas

In [ ]:
import matplotlib.pyplot as plt
counts = df.cls.value_counts().reindex(range(3), fill_value=0)
for c in range(3):
    print(f"{CLASS_NAMES[c]:8s}: {counts[c]:4d} ({100 * counts[c] / len(df):.1f}%)")
print(df.depth_mm.describe().round(1))

fig, ax = plt.subplots(figsize=(3.4, 2.3))
ax.hist(df.depth_mm, bins=40, color="#4477AA")
for t in THRESH_MM:
    ax.axvline(t, color="k", ls="--", lw=1)
ax.set_xlabel("Pothole depth (mm)", fontsize=8); ax.set_ylabel("Number of potholes", fontsize=8)
ax.tick_params(labelsize=8)
fig.tight_layout(); fig.savefig(f"{OUT}/depth_distribution.png", dpi=300); plt.show()
df.drop(columns="bbox").to_csv(f"{OUT}/depth_per_pothole.csv", index=False)

## 4. Bagi data train / val / test (70 / 15 / 15, terstratifikasi, seed tetap)

In [ ]:
import yaml
SEED = 42
DS = f"{WORK}/dataset"
if os.path.isdir(DS):
    shutil.rmtree(DS)

img_cls = df.groupby("image").cls.max()      # stratifikasi per gambar: kelas terdalam di gambar itu
rng = np.random.default_rng(SEED)
split = {}
for c in sorted(img_cls.unique()):
    names = rng.permutation(img_cls[img_cls == c].index.to_numpy())
    n_val = n_test = max(1, round(0.15 * len(names)))
    for j, nme in enumerate(names):
        split[nme] = "test" if j < n_test else "val" if j < n_test + n_val else "train"

for sp in ["train", "val", "test"]:
    os.makedirs(f"{DS}/images/{sp}", exist_ok=True); os.makedirs(f"{DS}/labels/{sp}", exist_ok=True)
for nme, g in df.groupby("image"):
    sp = split[nme]
    shutil.copy(rgb[nme], f"{DS}/images/{sp}/{os.path.basename(rgb[nme])}")
    with open(f"{DS}/labels/{sp}/{nme}.txt", "w") as f:
        for _, r in g.iterrows():
            f.write(f"{r.cls} " + " ".join(f"{v:.6f}" for v in r.bbox) + "\n")

DATA_YAML = f"{DS}/data.yaml"
yaml.safe_dump(dict(path=DS, train="images/train", val="images/val", test="images/test",
                    names=dict(enumerate(CLASS_NAMES))), open(DATA_YAML, "w"))

df["split"] = df.image.map(split)
tab = df.groupby(["split", "cls"]).size().unstack(fill_value=0).rename(columns=dict(enumerate(CLASS_NAMES)))
tab["images"] = pd.Series(split).value_counts()
tab.to_csv(f"{OUT}/split_summary.csv"); print(tab)

## 5. Training (tiga model, pengaturan sama)

In [ ]:
from ultralytics import YOLO, RTDETR

RUNS = f"{WORK}/runs"
MODELS = {
    "YOLO26s":   (YOLO,   "yolo26s.pt"),
    "YOLO26m":   (YOLO,   "yolo26m.pt"),
    "RT-DETR-L": (RTDETR, "rtdetr-l.pt"),
}
EPOCHS = 100
COMMON = dict(data=DATA_YAML, epochs=EPOCHS, patience=30, imgsz=640, batch=8, seed=SEED, device=0,
              project=RUNS, exist_ok=True)

if PREV_RUNS:
    for name in MODELS:
        dst = f"{RUNS}/{name}"
        if not os.path.exists(f"{dst}/results.png") and os.path.exists(f"{PREV_RUNS}/{name}/results.png"):
            shutil.copytree(f"{PREV_RUNS}/{name}", dst, dirs_exist_ok=True)
            print(name, "diambil dari versi sebelumnya")

for name, (cls, weights) in MODELS.items():
    last = f"{RUNS}/{name}/weights/last.pt"
    if os.path.exists(f"{RUNS}/{name}/results.png"):      # file ini baru dibuat saat training tuntas
        print(name, "sudah tuntas, dilewati")
    elif os.path.exists(last):
        print(name, "belum tuntas, dilanjutkan"); cls(last).train(resume=True)
    else:
        cls(weights).train(name=name, **COMMON)

## 6. Evaluasi di test set

In [ ]:
assert torch.cuda.is_available(), "GPU tidak aktif, evaluasi dibatalkan supaya angka kecepatan tidak salah."
rows, per_class, train_info = [], [], []
for name, (cls, _) in MODELS.items():
    log = pd.read_csv(f"{RUNS}/{name}/results.csv"); log.columns = [c.strip() for c in log.columns]
    best = log.loc[log["metrics/mAP50-95(B)"].idxmax()]
    train_info.append(dict(model=name, epochs_run=int(log.epoch.max()), epochs_planned=EPOCHS,
                           best_epoch=int(best.epoch), best_val_mAP50_95=round(float(best["metrics/mAP50-95(B)"]), 4),
                           train_minutes=round(float(log.time.max()) / 60, 1),
                           finished=os.path.exists(f"{RUNS}/{name}/results.png")))

    m = cls(f"{RUNS}/{name}/weights/best.pt")
    met = m.val(data=DATA_YAML, split="test", imgsz=640, batch=1, device=0, project=RUNS,
                name=f"{name}_test", exist_ok=True)
    ms = met.speed["inference"]
    rows.append(dict(model=name, params_M=sum(p.numel() for p in m.model.parameters()) / 1e6,
                     precision=met.box.mp, recall=met.box.mr, mAP50=met.box.map50, mAP50_95=met.box.map,
                     inference_ms=ms, fps=1000 / ms))
    for k, c in enumerate(met.box.ap_class_index):
        per_class.append(dict(model=name, cls=CLASS_NAMES[c], precision=met.box.p[k], recall=met.box.r[k],
                              AP50=met.box.ap50[k], AP50_95=met.box.ap[k]))

res = pd.DataFrame(rows).round(4); pc = pd.DataFrame(per_class).round(4); ti = pd.DataFrame(train_info)
res.to_csv(f"{OUT}/results_overall.csv", index=False); pc.to_csv(f"{OUT}/results_per_class.csv", index=False)
ti.to_csv(f"{OUT}/training_summary.csv", index=False)
print(ti.to_string(index=False)); print(); print(res.to_string(index=False)); print(); print(pc.to_string(index=False))

## 7. Kumpulkan hasil

File `pothole_results.zip` muncul di panel **Output** (`/kaggle/working`). Unduh dan kirim ke Claude.

In [ ]:
for name in MODELS:
    for sub in [name, f"{name}_test"]:
        files = sum([glob.glob(f"{RUNS}/{sub}/{pat}") for pat in ("*.png", "*.jpg", "*.csv", "args.yaml")], [])
        for p in files:
            os.makedirs(f"{OUT}/{sub}", exist_ok=True); shutil.copy(p, f"{OUT}/{sub}/")
with open(f"{OUT}/environment.txt", "w") as f:
    f.write(f"ultralytics {ultralytics.__version__}\ntorch {torch.__version__}\n"
            f"GPU {torch.cuda.get_device_name(0)}\nthresholds_mm {THRESH_MM}\nmax_valid_mm {MAX_VALID_MM}\nseed {SEED}\n")
shutil.make_archive(f"{WORK}/pothole_results", "zip", OUT)
print("Selesai:", f"{WORK}/pothole_results.zip")

## 8. Uji ketahanan terhadap kondisi buatan

Data uji yang sama diberi efek **gelap, silau, hujan, kabut, dan buram**. Labelnya tetap sama karena lubangnya sama,
hanya tampilannya yang diubah. Ini **simulasi**, bukan foto hujan/malam asli.

In [ ]:
import random
RB = f"{WORK}/robust"; RB_OUT = f"{WORK}/robust_out"
for p in (RB, RB_OUT):
    if os.path.isdir(p):
        shutil.rmtree(p)
os.makedirs(RB_OUT)

def gelap(im, rng):      # malam / mendung: kecerahan turun + derau sensor
    x = (im.astype(np.float32) / 255.0) ** 1.8 * 0.45 * 255
    x += rng.normal(0, 6, im.shape)
    return np.clip(x, 0, 255).astype(np.uint8)

def silau(im, rng):      # matahari terik / overexposure
    return np.clip(im.astype(np.float32) * 1.6 + 45, 0, 255).astype(np.uint8)

def kabut(im, rng):      # kabut / asap: dicampur abu-abu terang
    return np.clip(im.astype(np.float32) * 0.55 + 200 * 0.45, 0, 255).astype(np.uint8)

def buram(im, rng):      # gerak kendaraan: motion blur horizontal 11 px
    k = np.zeros((11, 11), np.float32); k[5, :] = 1 / 11
    return cv2.filter2D(im, -1, k)

def hujan(im, rng):      # garis hujan + sedikit gelap & buram
    h, w = im.shape[:2]
    layer = np.zeros((h, w), np.float32)
    for _ in range(int(h * w / 600)):
        x, y = int(rng.integers(0, w)), int(rng.integers(0, h))
        L = int(rng.integers(10, 25))
        cv2.line(layer, (x, y), (x + L // 5, y + L), 1.0, 1)
    layer = cv2.GaussianBlur(layer, (3, 3), 0)[..., None]
    x = im.astype(np.float32) * 0.75
    x = x * (1 - 0.6 * layer) + 220 * 0.6 * layer
    x = cv2.GaussianBlur(x, (3, 3), 0)
    return np.clip(x, 0, 255).astype(np.uint8)

CONDS = {"asli": None, "gelap": gelap, "silau": silau, "hujan": hujan, "kabut": kabut, "buram": buram}
test_imgs = sorted(glob.glob(f"{DS}/images/test/*"))
for cond, fn in CONDS.items():
    d = f"{RB}/{cond}"
    os.makedirs(f"{d}/images/test"); shutil.copytree(f"{DS}/labels/test", f"{d}/labels/test")
    rng = np.random.default_rng(SEED)
    for p in test_imgs:
        im = cv2.imread(p)
        cv2.imwrite(f"{d}/images/test/{os.path.basename(p)}", im if fn is None else fn(im, rng))
    yaml.safe_dump(dict(path=d, train="images/test", val="images/test", test="images/test",
                        names=dict(enumerate(CLASS_NAMES))), open(f"{d}/data.yaml", "w"))
print(len(test_imgs), "citra uji x", len(CONDS), "kondisi")

# contoh visual: satu citra dalam semua kondisi
ex = test_imgs[0]; tiles = []
for cond in CONDS:
    t = cv2.resize(cv2.imread(f"{RB}/{cond}/images/test/{os.path.basename(ex)}"), (320, 240))
    cv2.putText(t, cond, (8, 28), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (255, 255, 255), 4)
    cv2.putText(t, cond, (8, 28), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 0, 0), 2)
    tiles.append(t)
grid = np.vstack([np.hstack(tiles[:3]), np.hstack(tiles[3:])])
cv2.imwrite(f"{RB_OUT}/robustness_examples.jpg", grid)
plt.figure(figsize=(10, 5)); plt.imshow(grid[..., ::-1]); plt.axis("off"); plt.show()

## 9. Training ulang dengan augmentasi kuat

Model baseline memakai augmentasi bawaan Ultralytics (mosaic, flip horizontal, geser, skala, HSV).
Di sini ketiga model dilatih ulang dengan tambahan:
- **augmentasi cuaca offline**: separuh citra latih mendapat salinan bergaya gelap/silau/hujan/kabut/buram (acak, seed berbeda dari data uji);
- **augmentasi online lebih kuat**: kecerahan hsv_v 0,6, rotasi 5 derajat, skala 0,6, mixup 0,1.

Data validasi dan uji tidak diubah. Perkiraan 3-4 jam. Set `RUN_AUG_TRAINING = False` untuk melewati.

In [ ]:
RUN_AUG_TRAINING = True
AUG_PARAMS = dict(hsv_v=0.6, degrees=5.0, scale=0.6, mixup=0.1)
DS_AUG = f"{WORK}/dataset_aug"
AUG_MODELS = {}
if RUN_AUG_TRAINING:
    if not os.path.isdir(DS_AUG):
        for sp in ["train", "val", "test"]:
            shutil.copytree(f"{DS}/images/{sp}", f"{DS_AUG}/images/{sp}")
            shutil.copytree(f"{DS}/labels/{sp}", f"{DS_AUG}/labels/{sp}")
        rng_aug = np.random.default_rng(SEED + 1000)
        fx = [gelap, silau, hujan, kabut, buram]
        tr = sorted(glob.glob(f"{DS}/images/train/*")); n_aug = 0
        for p in tr:
            if rng_aug.random() < 0.5:
                stem, ext = os.path.splitext(os.path.basename(p))
                f = fx[int(rng_aug.integers(len(fx)))]
                cv2.imwrite(f"{DS_AUG}/images/train/{stem}_aug{ext}", f(cv2.imread(p), rng_aug))
                shutil.copy(f"{DS}/labels/train/{stem}.txt", f"{DS_AUG}/labels/train/{stem}_aug.txt")
                n_aug += 1
        print("citra latih:", len(tr), "+", n_aug, "salinan cuaca")
    DATA_YAML_AUG = f"{DS_AUG}/data.yaml"
    yaml.safe_dump(dict(path=DS_AUG, train="images/train", val="images/val", test="images/test",
                        names=dict(enumerate(CLASS_NAMES))), open(DATA_YAML_AUG, "w"))

    for name, (cls, weights) in MODELS.items():
        an = f"{name}-aug"
        AUG_MODELS[an] = (cls, weights)
        if PREV_RUNS and not os.path.exists(f"{RUNS}/{an}/results.png") and os.path.exists(f"{PREV_RUNS}/{an}/results.png"):
            shutil.copytree(f"{PREV_RUNS}/{an}", f"{RUNS}/{an}", dirs_exist_ok=True)
        if os.path.exists(f"{RUNS}/{an}/results.png"):
            print(an, "sudah tuntas, dilewati")
        elif os.path.exists(f"{RUNS}/{an}/weights/last.pt"):
            print(an, "belum tuntas, dilanjutkan"); cls(f"{RUNS}/{an}/weights/last.pt").train(resume=True)
        else:
            cls(weights).train(name=an, **{**COMMON, "data": DATA_YAML_AUG}, **AUG_PARAMS)

    aug_rows, aug_pc = [], []
    for an, (cls, _) in AUG_MODELS.items():
        m = cls(f"{RUNS}/{an}/weights/best.pt")
        met = m.val(data=DATA_YAML, split="test", imgsz=640, batch=1, device=0, project=RUNS,
                    name=f"{an}_test", exist_ok=True)
        ms = met.speed["inference"]
        aug_rows.append(dict(model=an, precision=met.box.mp, recall=met.box.mr, mAP50=met.box.map50,
                             mAP50_95=met.box.map, inference_ms=ms, fps=1000 / ms))
        for k, c in enumerate(met.box.ap_class_index):
            aug_pc.append(dict(model=an, cls=CLASS_NAMES[c], precision=met.box.p[k], recall=met.box.r[k],
                               AP50=met.box.ap50[k], AP50_95=met.box.ap[k]))
    pd.DataFrame(aug_rows).round(4).to_csv(f"{RB_OUT}/results_overall_aug.csv", index=False)
    pd.DataFrame(aug_pc).round(4).to_csv(f"{RB_OUT}/results_per_class_aug.csv", index=False)
    print(pd.DataFrame(aug_rows).round(4).to_string(index=False))

## 10. Uji ketahanan: semua model x semua kondisi

In [ ]:
ALL_MODELS = {**MODELS, **AUG_MODELS}
rob, rob_pc = [], []
for name, (cls, _) in ALL_MODELS.items():
    m = cls(f"{RUNS}/{name}/weights/best.pt")
    for cond in CONDS:
        met = m.val(data=f"{RB}/{cond}/data.yaml", split="test", imgsz=640, batch=8, device=0,
                    project=f"{WORK}/robust_runs", name=f"{name}_{cond}", exist_ok=True, plots=False, verbose=False)
        rob.append(dict(model=name, kondisi=cond, precision=met.box.mp, recall=met.box.mr,
                        mAP50=met.box.map50, mAP50_95=met.box.map))
        for k, c in enumerate(met.box.ap_class_index):
            rob_pc.append(dict(model=name, kondisi=cond, cls=CLASS_NAMES[c], AP50=met.box.ap50[k], AP50_95=met.box.ap[k]))

rob = pd.DataFrame(rob).round(4)
rob_pc = pd.DataFrame(rob_pc).round(4)
base = rob[rob.kondisi == "asli"].set_index("model")["mAP50"]
rob["turun_mAP50"] = (rob.mAP50 - rob.model.map(base)).round(4)
rob.to_csv(f"{RB_OUT}/robustness_overall.csv", index=False)
rob_pc.to_csv(f"{RB_OUT}/robustness_per_class.csv", index=False)
print(rob.pivot(index="kondisi", columns="model", values="mAP50").loc[list(CONDS)].to_string())

## 11. Satu citra, banyak lubang dengan kategori berbeda

Citra uji yang memuat dua lubang atau lebih dengan kategori berbeda. Tiap lubang dicocokkan dengan prediksi
(IoU >= 0,5) lalu dicek apakah kategorinya benar.

In [ ]:
import json
from collections import defaultdict


def iou(a, b):
    x0, y0 = max(a[0], b[0]), max(a[1], b[1])
    x1, y1 = min(a[2], b[2]), min(a[3], b[3])
    inter = max(0, x1 - x0) * max(0, y1 - y0)
    return inter / ((a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter + 1e-9)


def gt_boxes(img_path):
    h, w = cv2.imread(img_path).shape[:2]
    stem = os.path.splitext(os.path.basename(img_path))[0]
    out = []
    for line in open(f"{DS}/labels/test/{stem}.txt"):
        c, cx, cy, bw, bh = map(float, line.split())
        out.append((int(c), [(cx - bw / 2) * w, (cy - bh / 2) * h, (cx + bw / 2) * w, (cy + bh / 2) * h]))
    return out


multi = [p for p in test_imgs if len({c for c, _ in gt_boxes(p)}) > 1]
print("citra uji dengan >= 2 kategori berbeda:", len(multi), "| total lubang:", sum(len(gt_boxes(p)) for p in multi))

mp_rows, preds_cache = [], defaultdict(dict)
for name, (cls, _) in ALL_MODELS.items():
    m = cls(f"{RUNS}/{name}/weights/best.pt")
    for p in multi:
        r = m.predict(p, imgsz=640, conf=0.25, device=0, verbose=False)[0]
        pb = [(int(c), b.tolist(), float(s)) for c, b, s in zip(r.boxes.cls, r.boxes.xyxy, r.boxes.conf)]
        preds_cache[name][p] = pb
        used = set()
        for gi, (gc, gb) in enumerate(gt_boxes(p)):
            cand = [(iou(gb, b), j) for j, (_, b, _) in enumerate(pb) if j not in used]
            best = max(cand, default=(0, None))
            pc_ = None
            if best[0] >= 0.5:
                used.add(best[1]); pc_ = pb[best[1]][0]
            mp_rows.append(dict(model=name, image=os.path.basename(p), lubang=gi, aktual=CLASS_NAMES[gc],
                                prediksi=CLASS_NAMES[pc_] if pc_ is not None else "terlewat", benar=pc_ == gc))

mp = pd.DataFrame(mp_rows)
mp.to_csv(f"{RB_OUT}/multi_pothole_detail.csv", index=False)
summ = mp.groupby("model").agg(lubang=("benar", "size"), benar=("benar", "sum"),
                               terlewat=("prediksi", lambda s: (s == "terlewat").sum()))
summ["akurasi_kategori"] = (summ.benar / summ.lubang).round(3)
summ.to_csv(f"{RB_OUT}/multi_pothole_summary.csv")
print(summ.to_string())

# gambar contoh: maks 4 citra; kolom = label asli + 3 model baseline
COL = {0: (80, 200, 80), 1: (0, 200, 255), 2: (60, 60, 230)}
NM = {0: "dangkal", 1: "sedang", 2: "dalam"}


def draw(img, boxes, gt=False):
    im = img.copy()
    for b in boxes:
        c, (x0, y0, x1, y1) = b[0], map(int, b[1])
        cv2.rectangle(im, (x0, y0), (x1, y1), COL[c], 3)
        txt = NM[c] if gt else f"{NM[c]} {b[2]:.2f}"
        org = (x0 + 3, max(y0 - 6, 18))
        cv2.putText(im, txt, org, cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 4)
        cv2.putText(im, txt, org, cv2.FONT_HERSHEY_SIMPLEX, 0.7, COL[c], 2)
    return cv2.resize(im, (400, 300))


show = sorted(multi, key=lambda p: -len({c for c, _ in gt_boxes(p)}))[:4]
rows_img = []
for p in show:
    img = cv2.imread(p)
    rows_img.append(np.hstack([draw(img, gt_boxes(p), gt=True)] + [draw(img, preds_cache[n][p]) for n in MODELS]))
if rows_img:
    head = np.full((40, rows_img[0].shape[1], 3), 255, np.uint8)
    for i, t in enumerate(["Label asli"] + list(MODELS)):
        cv2.putText(head, t, (i * 400 + 10, 28), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 0), 2)
    fig_mp = np.vstack([head] + rows_img)
    cv2.imwrite(f"{RB_OUT}/multi_pothole_examples.jpg", fig_mp)
    for p in show:   # simpan juga citra aslinya untuk gambar paper
        shutil.copy(p, f"{RB_OUT}/multi_src_{os.path.basename(p)}")
    with open(f"{RB_OUT}/multi_pothole_boxes.json", "w") as f:
        json.dump({os.path.basename(p): dict(gt=gt_boxes(p), pred={n: preds_cache[n][p] for n in ALL_MODELS})
                   for p in show}, f)
    plt.figure(figsize=(14, 3.5 * len(rows_img))); plt.imshow(fig_mp[..., ::-1]); plt.axis("off"); plt.show()

## 12. Kumpulkan hasil tambahan

Unduh `pothole_robustness.zip` dari tab **Output** dan kirim ke Claude.

In [ ]:
with open(f"{RB_OUT}/robustness_settings.txt", "w") as f:
    f.write(f"seed {SEED}\nmodel dari versi sebelumnya: {PREV_RUNS}\n"
            "gelap: gamma 1.8, x0.45, derau N(0,6)\nsilau: x1.6 + 45\nkabut: 0.55*citra + 0.45*200\n"
            "buram: motion blur horizontal 11 px\n"
            "hujan: garis (h*w/600 buah, panjang 10-25 px) dicampur 0.6, x0.75, Gaussian 3x3\n"
            f"augmentasi kuat: offline cuaca p=0.5 (seed {SEED + 1000}), online {AUG_PARAMS}\n")
for an in AUG_MODELS:
    for sub in [an, f"{an}_test"]:
        for pat in ("*.png", "*.jpg", "*.csv", "args.yaml"):
            for p in glob.glob(f"{RUNS}/{sub}/{pat}"):
                os.makedirs(f"{RB_OUT}/{sub}", exist_ok=True)
                shutil.copy(p, f"{RB_OUT}/{sub}/")
shutil.copy(f"{OUT}/results_overall.csv", f"{RB_OUT}/results_overall_baseline_rerun.csv")
shutil.copy(f"{OUT}/results_per_class.csv", f"{RB_OUT}/results_per_class_baseline_rerun.csv")
shutil.make_archive(f"{WORK}/pothole_robustness", "zip", RB_OUT)
print("Selesai:", f"{WORK}/pothole_robustness.zip")